# RC Stable Audio Tools - Cloud GPU

Use **Runtime > Change runtime type > GPU** (T4) before running.

**Cell 1** will detect if Colab's default PyTorch is CPU-only and automatically reinstall the CUDA build, then restart the runtime. After the restart, run all cells again.

In [ ]:
# Colab now ships with cpu-only PyTorch by default.
# This cell reinstalls the CUDA build and restarts the runtime so the GPU is available.
import torch, os
if not torch.cuda.is_available():
    import subprocess
    print('CPU-only torch detected — reinstalling CUDA build...')
    subprocess.run(
        ['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        check=True
    )
    print('Done. Restarting runtime — re-run all cells after restart.')
    os.kill(os.getpid(), 9)
else:
    print(f'CUDA already available: {torch.cuda.get_device_name(0)}')


In [ ]:
import sys, platform, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No CUDA GPU detected. In Colab, switch Runtime > Change runtime type > GPU and restart.')

## Clone the patched branch

In [ ]:
%cd /content
!rm -rf RC-stable-audio-tools
!git clone -b codex-cloud-gpu-ready https://github.com/ebk5k/RC-stable-audio-tools.git
%cd /content/RC-stable-audio-tools
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

## Install dependencies, MIDI support, and model

This runs the repo-owned Colab bootstrap script. It installs the cloud dependency set, adds the Basic Pitch ONNX MIDI backend, installs this repo, and downloads Foundation-1.

In [ ]:
!bash scripts/colab_bootstrap.sh


## Re-check CUDA after install

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Confirm the Foundation-1 model is present

In [ ]:
!find models/RoyalCities-Foundation-1 -maxdepth 1 -type f | sort

## Launch Gradio

Open the public `gradio.live` link printed by this cell. Keep the cell running while you use the app.

In [ ]:
!python run_gradio.py --share

## Download generations back to your computer

Run this after you generate files.

In [ ]:
!zip -r generations.zip generations
from google.colab import files
files.download('generations.zip')